<a href="https://colab.research.google.com/github/zhaoyingjun/Tiny-R2/blob/main/Tiny_R2_journey.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tiktoken datasets transformers huggingface_hub
!pip install git+https://github.com/KellerJordan/Muon
!pip install --upgrade transformers


In [ ]:
!pip install faiss-cpu sentence_transformers datasets rouge-score nltk rank_bm25 tqdm

In [ ]:
!hf auth login --force

/bin/bash: line 1: hf: command not found


In [9]:
!git clone https://github.com/zhaoyingjun/Tiny-R2.git
%cd Tiny-R2

Cloning into 'Tiny-R2'...
remote: Enumerating objects: 542, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 542 (delta 62), reused 9 (delta 9), pack-reused 445 (from 1)
Receiving objects: 100% (542/542), 4.01 MiB | 42.35 MiB/s, done.
Resolving deltas: 100% (321/321), done.
/content/Tiny-R2/Tiny-R2/Tiny-R2


**训练climbmix数据，可以训练一个基础大模型，1.7B参数量，评测模型结构和效果。支持采用大模型作为Agent进行观察训练过程并给出实施的lr、clip的调整参数;--use_agent_observe=True开启,默认是关闭的，需要输入gemini的api_key**

默认不开启Agent智能化自主训练

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True

开启Agent智能化自主训练，需要将your gemini api key替换

In [ ]:
!python train.py --n_layer 6 --n_embd 1536 --hc 'True' --mhc 'True' --n_experts 8 --max_iters 10000 --attention_types 'Sparse' --batch_size 8 --ctx_len 2048 --hf_dataset 'karpathy/climbmix-400b-shuffle' --resume True --save_best_only True --use_agent_observe True --gemini_api_key 'your gemini api key'

预训练完成之后，进行benchmark的验证

In [ ]:
!python /content/Tiny-R2/evaluate.py --checkpoint /content/Tiny-R2/checkpoints/best_model_step_1180.pt

后训练直接使用Qwen3.5-9B模型作为教师模型进行OPD训练，可自定义使用RAG增加教师模型以及自定义数据集等

In [ ]:
!python opd_train.py --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name tiny-r2 \
  --student_ckpt checkpoints/best_model_step_40.pt \
  --batch_size 2 \
  --grad_accum_steps 4 \
  --custom_qa_path baoxianqa.jsonl \
  --rag_corpus_path baoxianqa.txt

使用Qwen3.5-9B模型作为教师模型、Qwen3.5-0.8B作为学生模型进行OPD训练，用RAG增加教师模型，RAG数据集来自问答数据集集medquad，可以复现Readme中的结果;去除命令行中--enable_reg_teacher则关闭外挂RAG功能；--rag_corpus_path外挂RAG数据集，--custom_qa_path 自定义问题集

In [ ]:
!python opd_train.py \
  --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name Qwen/Qwen3.5-0.8B \
  --batch_size 2 \
  --grad_accum_steps 4\
  --student_use_rag


In [ ]:
!python opd_train_utral.py \
  --hf_teacher_model Qwen/Qwen3.5-9B \
  --student_model_name Qwen/Qwen3.5-0.8B \
  --batch_size 1 \
  --grad_accum_steps 1\
  --student_use_rag\
  --enable_wandb

✅ 成功导入 rank_bm25 与 sentence_transformers，混合 RAG 模块已启用。
✅ 成功导入 Tiny-R2 核心模块

🚀 Tiny-R2 OPD 深度在策表征对齐 (Advantage-Weighted Group-OPD) 训练流水线启动
📦 本地缓存存储: ./hf_cache

config.json: 100% 3.13k/3.13k [00:00<00:00, 19.5MB/s]
tokenizer_config.json: 100% 16.7k/16.7k [00:00<00:00, 90.1MB/s]
vocab.json: 100% 6.72M/6.72M [00:00<00:00, 168MB/s]
merges.txt: 100% 3.35M/3.35M [00:00<00:00, 209MB/s]
tokenizer.json: 100% 12.8M/12.8M [00:00<00:00, 24.5MB/s]
chat_template.jinja: 100% 7.76k/7.76k [00:00<00:00, 45.7MB/s]
📡 载入 Hugging Face 线上数据集: pubmed_qa
Generating train split: 100% 1000/1000 [00:00<00:00, 141536.88 examples/s]
🔒 正在构建标准 RAG 语料库...
[-] 检索语料库构建完成，共包含 1000 条文档。
[-] 正在初始化过滤后的 BM25 词法索引...
[-] 正在初始化密向量检索模型 (BAAI/bge-small-en-v1.5)...
modules.json: 100% 349/349 [00:00<00:00, 4.57MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 1.72MB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 227MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 599kB/s]
config.json: 100% 743/743 [00:

默认使用medquade数据集进行OPD的结果验证，可以根据OPD训练的数据集对应的修改验证

In [ ]:
!python eval_opd.py \
    --dataset medquad \
    --hf_teacher_model Qwen/Qwen3.5-9B \
    --student_base_model Qwen/Qwen3.5-0.8B \
    --student_opd_model opd_checkpoints/student_best_model_step_450.pt \
    --eval_samples 100